In [ ]:
# Copyright 2025 DeepMind Technologies Limited. All Rights Reserved.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/google-gemini/genai-processors/blob/main/notebooks/function_calling.ipynb)
[![License](https://img.shields.io/badge/License-Apache_2.0-blue.svg)](https://github.com/google-gemini/genai-processors/blob/main/LICENSE)

# Function Calling with Processors 📞

The GenAI Processor library provides a processor to run Function Calls with any
model that can produce GenAI FunctionCall parts. This notebook provides a
step-by-step guide on how to do this.



## 1. 🛠️ Setup

First, install the GenAI Processors library:

In [ ]:
!pip install genai-processors

### API Key

You will need an API key to call the Genai models in this notebook. If you have
not done so already, obtain your API key from Google AI Studio, and import it as
a secret in Colab (recommended) or directly set it below.

In [ ]:
from google.colab import userdata

API_KEY = userdata.get('GOOGLE_API_KEY')

## 2. 👆 Define the functions to be called

The Tool definition will be automatically derived from the function signature.
So it is advised to have a good docstring that covers the arguments and the
returned value. We use introspection though and the model will see the function
schema even if docstring is missing.

In [ ]:
def get_weather(location: str) -> str:
  """Returns the weather information.

  The weather information will contain the temperature in Celsius.

  Args:
    location: name of the city, region or place where the weather is requested.

  Returns:
    a string describing the weather at the provided location.
  """
  return f'Weather in {location} is 21 degrees celcius and rainy'


def to_fahrenheit(celsius: int) -> str:
  """Returns fahrenheit temperature given the value in celsius."""
  fahrenheit = (celsius * 9 / 5) + 32
  return fahrenheit

## 3. ✨ Create a function calling processor

We will use the Gemini API for our model. Tool declarations and disabling of
automatic function calling are handled automatically by the `FunctionCalling`
processor via `add_tools()` — you only need to pass your functions once.

If you choose other models (e.g. Ollama, Transformers), the same pattern
applies: just wrap the model with `FunctionCalling(fns=...)`.

In [ ]:
from genai_processors.core import genai_model

# Initialize the GenAI model processor
# Replace 'gemini-2.5-flash' with your desired model name
genai_processor = genai_model.GenaiModel(
    api_key=API_KEY,
    model_name="gemini-2.5-flash",
)

Then adding function calling is done as follows:

In [ ]:
from genai_processors.core import function_calling

fc = function_calling.FunctionCalling(
    genai_processor,
    fns=[get_weather, to_fahrenheit],
)

## 4. ▶️ Run the function calling processor

The function calling processor is typically used on a stream of user prompts.

In [ ]:
from genai_processors import streams
import nest_asyncio

nest_asyncio.apply()  # Needed to run async loops in Colab

async for part in fc(
    'What is the weather in Cherbourg? Give the temperature in Fahrenheit.'
):
  if not part.substream_name:
    # default substream - contains what the user would see.
    print(f'\033[0m {part.text}', flush=True, end='')

  if part.substream_name:
    # subtream_name = "function_call" / internal function calls.
    if part.function_call:
      print(
          f'\033[96m FC: {part.function_call.name}: {part.function_call.args}',
          flush=True,
      )
    elif part.function_response:
      print(
          f'\033[96m FR: {part.function_response.response}',
          flush=True,
      )

## 5. 🔗 Nesting FunctionCalling Processors

`FunctionCalling` processors can be **nested**. When a model returns a function
call whose name is not in the inner processor's `fns` list, the call is passed
through unmodified to the output stream—no error is raised. An outer
`FunctionCalling` processor can then intercept and execute it.

This enables hierarchical agent patterns—for example, a supervisor agent that
owns high-level tools while a subordinate agent handles domain-specific tools.
Because tool declarations are automatically registered on the model by
`FunctionCalling`, you **do not** need to define tools twice across nested
processors.

```python
# Inner agent handles weather lookups.
inner_agent = function_calling.FunctionCalling(genai_processor, fns=[get_weather])

# Outer agent handles temperature conversion.
# If the model calls `get_weather`, the inner processor handles it.
# If the model calls `to_fahrenheit`, the inner processor passes it through
# and the outer processor intercepts it.
outer_agent = function_calling.FunctionCalling(inner_agent, fns=[to_fahrenheit])
```